# Urdu-English Machine Translation (A3 Project)

**Student**: Htut Ko Ko  
**Course**: Natural Language Understanding  
**Task**: Urdu (ur) <-> English (en) Translation using Transformer

## Project Overview
This notebook implements a Neural Machine Translation system using a **Transformer** architecture.
We use the **Opus-100** dataset for Urdu-English parallel data.
We use **SentencePiece** for subword tokenization.


In [1]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import sentencepiece as spm

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

Using device: cuda


## 2. Data Loading (Opus-100)
Loading Urdu-English pairs from Opus-100.

In [2]:
print("Loading Opus-100 Dataset (Urdu-English)...")
try:
    # Opus-100 has 'en-ur'
    dataset = load_dataset("opus100", "en-ur", split="train+validation+test")
    print(f"Loaded {len(dataset)} sentences from Opus-100 dataset.")

    # Extract data
    data = []
    for item in dataset:
        if 'translation' in item:
            if 'ur' in item['translation'] and 'en' in item['translation']:
                data.append({
                    'ur': item['translation']['ur'],
                    'en': item['translation']['en']
                })

    # Limit to manageable size
    if len(data) > 50000:
        import random
        random.shuffle(data)
        data = data[:50000]
        print("Subsampled to 50,000 examples for efficiency.")

    print(f"Extracted {len(data)} Urdu-English pairs.")

except Exception as e:
    print(f"Error loading from HF: {e}")

Loading Opus-100 Dataset (Urdu-English)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en-ur/test-00000-of-00001.parquet:   0%|          | 0.00/301k [00:00<?, ?B/s]

en-ur/train-00000-of-00001.parquet:   0%|          | 0.00/148M [00:00<?, ?B/s]

en-ur/validation-00000-of-00001.parquet:   0%|          | 0.00/296k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/753913 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loaded 757913 sentences from Opus-100 dataset.
Subsampled to 50,000 examples for efficiency.
Extracted 50000 Urdu-English pairs.


In [3]:
df = pd.DataFrame(data)
print(df.head())

df = df.dropna(subset=['ur', 'en'])
df['ur'] = df['ur'].astype(str)
df['en'] = df['en'].astype(str)
df = df[df['ur'].str.strip() != '']
df = df[df['en'].str.strip() != '']

                                                  ur  \
0  اور وہ کہ جب کہ انہیں ان کے رب کی آیتیں یاد د ...   
1  یہ سارے کے سارے قیامت کے دن اکیلے اس کے پاس حا...   
2  شیطان ان سے وعدے کرتا ہےاور انہیں امیدیں دلاتا...   
3  کیا تم نے نہیں دیکھا کہ خدا نے سات آسمان کیسے ...   
4  لیکن ہم نے بہت سی نسلیں پیدا کیں جن پر لمبی مد...   

                                                  en  
0  Who, when reminded of their Lord's revelations...  
1  And everyone of them will come to Him alone on...  
2  He [Shaitan (Satan)] makes promises to them, a...  
3  "Have you not seen that God has created the se...  
4  But We raised up (new) generations, and long w...  


## 3. Tokenization

In [4]:
# Save texts to files
with open('train_ur.txt', 'w', encoding='utf-8') as f:
    for line in df['ur']: f.write(line + '\n')

with open('train_en_ur.txt', 'w', encoding='utf-8') as f:
    for line in df['en']: f.write(line + '\n')

# Train SentencePiece models
vocab_size = 8000
model_type = 'bpe'

print("Training Urdu Tokenizer...")
spm.SentencePieceTrainer.train(
    input='train_ur.txt',
    model_prefix='spm_ur',
    vocab_size=vocab_size,
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

print("Training English Tokenizer (for Urdu pair)...")
spm.SentencePieceTrainer.train(
    input='train_en_ur.txt',
    model_prefix='spm_en_ur',
    vocab_size=vocab_size,
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

sp_src = spm.SentencePieceProcessor(model_file='spm_ur.model')
sp_trg = spm.SentencePieceProcessor(model_file='spm_en_ur.model')

Training Urdu Tokenizer...
Training English Tokenizer (for Urdu pair)...


## 4. Dataset & Model

In [5]:
class TranslationDataset(Dataset):
    def __init__(self, df, sp_src, sp_trg):
        self.data = df
        self.sp_src = sp_src
        self.sp_trg = sp_trg

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text = self.data.iloc[idx]['ur']
        trg_text = self.data.iloc[idx]['en']
        src_ids = [self.sp_src.bos_id()] + self.sp_src.encode(src_text, out_type=int) + [self.sp_src.eos_id()]
        trg_ids = [self.sp_trg.bos_id()] + self.sp_trg.encode(trg_text, out_type=int) + [self.sp_trg.eos_id()]
        return torch.tensor(src_ids), torch.tensor(trg_ids)

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)
    src_pad = pad_sequence(src_batch, batch_first=True, padding_value=0)
    trg_pad = pad_sequence(trg_batch, batch_first=True, padding_value=0)
    return src_pad, trg_pad

train_dataset = TranslationDataset(df, sp_src, sp_trg)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), :]
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size,
                 d_model=256, nhead=4, num_encoder_layers=2,
                 num_decoder_layers=2, dim_feedforward=512, dropout=0.1, pad_idx=0):
        super(TransformerModel, self).__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(d_model=d_model, nhead=nhead, num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(d_model, trg_vocab_size)

    def forward(self, src, trg):
        src_key_padding_mask = (src == self.pad_idx)
        trg_mask = self.transformer.generate_square_subsequent_mask(trg.size(1)).to(src.device)
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        trg_emb = self.pos_encoder(self.trg_embedding(trg) * math.sqrt(self.d_model))
        output = self.transformer(src=src_emb, tgt=trg_emb, tgt_mask=trg_mask, src_key_padding_mask=src_key_padding_mask)
        return self.fc_out(output)

In [7]:
model = TransformerModel(vocab_size, vocab_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Starting Training...")
for epoch in range(10): # 10 Epochs for demo
    model.train()
    epoch_loss = 0
    for i, (src, trg) in enumerate(train_loader):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg[:, :-1])
        output = output.contiguous().view(-1, output.shape[-1])
        trg = trg[:, 1:].contiguous().view(-1)
        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        if i % 100 == 0: print(f"Step {i}, Loss: {loss.item():.3f}")
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.3f}")

    # Save
    torch.save(model.state_dict(), 'transformer_model_ur.pt')

Starting Training...
Step 0, Loss: 9.226
Step 100, Loss: 5.868
Step 200, Loss: 5.375
Step 300, Loss: 5.134
Step 400, Loss: 4.910
Step 500, Loss: 4.746
Step 600, Loss: 4.627
Step 700, Loss: 4.644
Epoch 1 Loss: 5.131
Step 0, Loss: 4.443
Step 100, Loss: 4.345
Step 200, Loss: 4.251
Step 300, Loss: 4.343
Step 400, Loss: 4.134
Step 500, Loss: 4.181
Step 600, Loss: 4.057
Step 700, Loss: 4.120
Epoch 2 Loss: 4.244
Step 0, Loss: 4.032
Step 100, Loss: 4.073
Step 200, Loss: 3.957
Step 300, Loss: 3.818
Step 400, Loss: 3.853
Step 500, Loss: 3.864
Step 600, Loss: 3.875
Step 700, Loss: 3.605
Epoch 3 Loss: 3.886
Step 0, Loss: 3.811
Step 100, Loss: 3.600
Step 200, Loss: 3.583
Step 300, Loss: 3.638
Step 400, Loss: 3.742
Step 500, Loss: 3.589
Step 600, Loss: 3.483
Step 700, Loss: 3.716
Epoch 4 Loss: 3.640
Step 0, Loss: 3.661
Step 100, Loss: 3.563
Step 200, Loss: 3.329
Step 300, Loss: 3.446
Step 400, Loss: 3.519
Step 500, Loss: 3.519
Step 600, Loss: 3.268
Step 700, Loss: 3.564
Epoch 5 Loss: 3.458
Step 0, L

In [8]:
# Copy to app
import shutil
os.makedirs('app/models', exist_ok=True)
shutil.copy('transformer_model_ur.pt', 'app/models/transformer_model_ur.pt')
shutil.copy('spm_ur.model', 'app/models/spm_ur.model')
shutil.copy('spm_en_ur.model', 'app/models/spm_en_ur.model')
print("Models copied to app/models/")

Models copied to app/models/
